# 01 · Reference points on TwoRoom: LeWM and full-grid DINO-WM

Evaluates two released models on the same 50 start/goal queries and with the same CEM planner, and
measures success rate and planning time per replan.

| model | source | tokens per frame |
|---|---|---|
| LeWM | `quentinll/lewm-tworooms` (official) | 1 |
| DINO-WM, no proprio | `fafraob/point-cloud-tworoom`, folder `dino-wm/` | 196 |

The DINO-WM checkpoint is **not** the original LeWM-paper baseline (that download is gone). It was retrained
by the authors of "Does Latent Planning Survive Point Clouds?" (arXiv:2608.29434) with the official `dino_wm`
code on the same TwoRoom dataset, and is loaded through their MIT-licensed adapter.

Without a CUDA GPU the notebook switches to a tiny smoke configuration (2 episodes, 16 samples, 2 CEM
iterations) that only tests the code path.

## 1. Paths and environment variables

In [1]:
import os
import shutil
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    ROOT = Path("/content")
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# must be set before importing stable_worldmodel / mujoco
os.environ["STABLEWM_HOME"] = str(ROOT / "stablewm")      # datasets/ and checkpoints/ live here
os.environ.setdefault("WANDB_MODE", "disabled")
if sys.platform == "linux":
    os.environ.setdefault("MUJOCO_GL", "egl")             # headless rendering; macOS uses its default

HOME = Path(os.environ["STABLEWM_HOME"])
(HOME / "datasets").mkdir(parents=True, exist_ok=True)
print("root:", ROOT)
print("STABLEWM_HOME:", HOME)
print(f"free disk: {shutil.disk_usage(HOME).free / 1e9:.0f} GB")

root: /content
STABLEWM_HOME: /content/stablewm
free disk: 207 GB


## 2. Install (Colab only)

Same install as notebook 00. If Colab asks to restart the runtime, restart and run from the top again.

In [2]:
PACKAGES = [
    "stable-worldmodel[train,env]==0.1.1",
    "hdf5plugin",
    "mujoco>=3.14,<3.15",
    "transformers>=4.50,<5",
    "datasets>=3",
    "huggingface-hub<1",
]

if IN_COLAB:
    import subprocess

    def pip(*args):
        cmd = [sys.executable, "-m", "uv", "pip", "install", "--system", *args]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            print((result.stdout + result.stderr)[-6000:])      # uv's own error message
            raise RuntimeError("install failed, see the output above")

    # Install system dependency swig required to compile box2d-py
    print("Installing system dependencies (swig)... ")
    subprocess.run(["apt-get", "update", "-y"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(["apt-get", "install", "-y", "swig"], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

    if sys.version_info >= (3, 13):
        stub = Path("/content/labmaze_stub")
        (stub / "labmaze").mkdir(parents=True, exist_ok=True)
        (stub / "labmaze" / "__init__.py").write_text("# empty stand-in, see notebook section 2\n")
        (stub / "pyproject.toml").write_text(
            "[build-system]\nrequires = [\"setuptools\"]\nbuild-backend = \"setuptools.build_meta\"\n"
            "[project]\nname = \"labmaze\"\nversion = \"1.0.6\"\n"
        )
        pip(str(stub))

    pip(*PACKAGES)
    print("installed on python", sys.version.split()[0])

Installing system dependencies (swig)... 
installed on python 3.13.15


## 3. Imports and settings

In [3]:
import csv
import datetime
import json
import platform
import subprocess
import time
import warnings
from importlib.metadata import version

import numpy as np
import torch

warnings.filterwarnings("ignore")
import stable_pretraining as spt
import stable_worldmodel as swm  # must be imported before the DINO-WM adapter (section 5)

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
GPU = torch.cuda.get_device_name(0) if DEVICE == "cuda" else DEVICE
SMOKE = DEVICE != "cuda"

ENV_NAME = "swm/TwoRoom-v1"
SEED = 42
N_EVAL = 2 if SMOKE else 50                  # LeWM
N_EVAL_DINOWM = 2 if SMOKE else 50          # full-grid DINO-WM takes minutes per replan on a T4, see section 7
GOAL_OFFSET, EVAL_BUDGET = 25, 50
PLAN = dict(horizon=5, receding_horizon=5, action_block=5)
CEM = dict(num_samples=16, n_steps=2, topk=4) if SMOKE else dict(num_samples=300, n_steps=30, topk=30)

RESULTS_DIR = ROOT / "results"
(RESULTS_DIR / "queries").mkdir(parents=True, exist_ok=True)

print("python", platform.python_version(), "| torch", torch.__version__, "| device", GPU)
print("smoke configuration" if SMOKE else "full configuration", "|", N_EVAL, "/", N_EVAL_DINOWM, "episodes | CEM", CEM)

11:29:42 | INFO  | __init__.py | NumExpr defaulting to 12 threads.
python 3.13.15 | torch 2.11.0+cu130 | device NVIDIA A100-SXM4-40GB
full configuration | 50 / 50 episodes | CEM {'num_samples': 300, 'n_steps': 30, 'topk': 30}


## 4. Dataset and the pinned start/goal queries

The queries are drawn once (same procedure and seed as LeWM's `eval.py`) and saved, so every model in this project sees identical episodes.

In [4]:
import tarfile
import urllib.request

import zstandard
from tqdm.auto import tqdm

TWOROOM_URL = "https://huggingface.co/datasets/quentinll/lewm-tworooms/resolve/main/tworoom.tar.zst"
MIN_FREE_GB = 20
LOCAL_EPISODES = 20


def download_and_extract(url, dest_dir):
    """Stream a .tar.zst from `url` and extract it into `dest_dir`."""
    with urllib.request.urlopen(url) as resp:
        total = int(resp.headers.get("Content-Length", 0)) or None
        with tqdm.wrapattr(resp, "read", total=total, desc=Path(url).name) as raw:
            with zstandard.ZstdDecompressor().stream_reader(raw) as zr:
                with tarfile.open(fileobj=zr, mode="r|") as tar:
                    tar.extractall(dest_dir)


def collect_local_tworoom(path, episodes):
    from stable_worldmodel.envs.two_room import ExpertPolicy

    world = swm.World("swm/TwoRoom-v1", num_envs=4, image_shape=(224, 224),
                      max_episode_steps=100, render_mode="rgb_array")
    world.set_policy(ExpertPolicy(action_noise=2.0, action_repeat_prob=0.05))
    world.collect(path, episodes=episodes, seed=0, format="hdf5")
    world.close()


free_gb = shutil.disk_usage(HOME).free / 1e9
if (HOME / "datasets" / "tworoom.h5").exists():
    DATASET = "tworoom"
elif free_gb >= MIN_FREE_GB:
    download_and_extract(TWOROOM_URL, HOME / "datasets")
    DATASET = "tworoom"
else:
    DATASET = "tworoom_local"
    if not (HOME / "datasets" / "tworoom_local.h5").exists():
        print(f"only {free_gb:.0f} GB free: collecting {LOCAL_EPISODES} local episodes instead")
        collect_local_tworoom(HOME / "datasets" / "tworoom_local.h5", LOCAL_EPISODES)

dataset = swm.data.HDF5Dataset(DATASET, keys_to_cache=["action", "proprio"])
lengths, offsets = np.asarray(dataset.lengths), np.asarray(dataset.offsets)
print(f"dataset: {DATASET} | {len(lengths)} episodes | {lengths.sum()} steps | columns: {dataset.column_names}")

tworoom.tar.zst:   0%|          | 0/3425937909 [00:00<?, ?it/s]

11:30:29 | INFO  | __init__.py | Cached 'action' from '/content/stablewm/datasets/tworoom.h5'
11:30:29 | INFO  | __init__.py | Cached 'proprio' from '/content/stablewm/datasets/tworoom.h5'
dataset: tworoom | 10000 episodes | 920809 steps | columns: ['action', 'distance_to_target', 'ep_idx', 'id', 'observation', 'pixels', 'pos_agent', 'pos_target', 'proprio', 'render_time', 'reward', 'step_idx', 'terminated', 'truncated']


In [5]:
row_episode = np.repeat(np.arange(len(lengths)), lengths)
row_step = np.arange(lengths.sum()) - np.repeat(offsets, lengths)
valid_rows = np.nonzero(row_step <= np.repeat(lengths, lengths) - GOAL_OFFSET - 1)[0]
rng = np.random.default_rng(SEED)
N_QUERIES = min(50, len(valid_rows) - 1)
rows = np.sort(valid_rows[rng.choice(len(valid_rows) - 1, size=N_QUERIES, replace=False)])
QUERIES = {"dataset": DATASET, "seed": SEED, "goal_offset": GOAL_OFFSET,
           "episodes": row_episode[rows].tolist(), "start_steps": row_step[rows].tolist()}

query_file = RESULTS_DIR / "queries" / f"{DATASET}_seed{SEED}.json"
query_file.write_text(json.dumps(QUERIES, indent=1))
print(f"{N_QUERIES} queries saved to {query_file}")
print("episodes:", QUERIES["episodes"])
print("start steps:", QUERIES["start_steps"])

50 queries saved to /content/results/queries/tworoom_seed42.json
episodes: [644, 687, 872, 905, 933, 953, 1287, 1661, 1832, 2019, 2280, 2773, 3558, 3719, 4032, 4339, 4398, 4445, 4467, 4515, 4516, 5014, 5141, 5272, 5459, 5552, 6320, 6446, 6552, 6791, 6988, 7021, 7189, 7370, 7597, 7627, 7751, 7797, 7826, 7873, 8241, 8289, 8411, 8598, 8601, 8895, 8949, 9277, 9711, 9757]
start steps: [42, 68, 25, 16, 20, 17, 72, 21, 10, 36, 50, 13, 57, 39, 0, 7, 52, 6, 38, 21, 0, 72, 7, 33, 55, 36, 28, 26, 41, 37, 59, 44, 16, 22, 55, 48, 27, 32, 66, 32, 30, 18, 21, 31, 39, 31, 49, 75, 8, 70]


## 5. Load both models

DINO-WM needs two extra things: the adapter repo (pinned commit, it also vendors the original `dino_wm`
code) and the checkpoint folder. DINOv2 ViT-S/14 itself comes from `torch.hub` on first use.

The checkpoint was trained with bf16 mixed precision. On CUDA it runs here with fp16 autocast: a T4 has fast
fp16 but no native bf16, and fp32 is 2.4x slower (613 s against 259 s per replan, measured on a T4). LeWM
runs in fp32; at about 2 s per replan it does not need it.

In [6]:
from torchvision.transforms import v2 as transforms

# --- LeWM (official)
lewm = swm.wm.utils.load_pretrained("quentinll/lewm-tworooms").to(DEVICE).eval()
lewm.requires_grad_(False)
lewm_transform = transforms.Compose([
    transforms.ToImage(),
    transforms.ToDtype(torch.float32, scale=True),
    transforms.Normalize(**spt.data.dataset_stats.ImageNet),
    transforms.Resize(size=224),
])

# --- DINO-WM (third-party retrain): adapter code
ADAPTER_URL = "https://github.com/fafraob/point-lewm.git"
ADAPTER_COMMIT = "d2ba2f7e3a1cacdddb3b2991dcc7bd463b733a48"
ADAPTER_DIR = ROOT / "third_party" / "point-lewm"
if not ADAPTER_DIR.exists():
    ADAPTER_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "-q", ADAPTER_URL, str(ADAPTER_DIR)], check=True)
subprocess.run(["git", "-C", str(ADAPTER_DIR), "checkout", "-q", ADAPTER_COMMIT], check=True)

# The vendored dino_wm code has a top-level package called `datasets`, the same name as Hugging Face
# `datasets`, which is already imported. Step the HF one aside while the adapter imports, then put it back.
DINO_WM_ROOT = str(ADAPTER_DIR / "third_party" / "dino_wm")
is_datasets = lambda k: k == "datasets" or k.startswith("datasets.")
hf_datasets = {k: sys.modules.pop(k) for k in list(sys.modules) if is_datasets(k)}
sys.path[:0] = [DINO_WM_ROOT, str(ADAPTER_DIR)]
from dinowm_swm.planning import load_dinowm_cost_model

for k in [k for k in sys.modules if is_datasets(k)]:
    del sys.modules[k]
sys.modules.update(hf_datasets)
sys.path.remove(DINO_WM_ROOT)
sys.path.append(DINO_WM_ROOT)            # still importable (the checkpoint unpickles `models.*`), but last

# --- DINO-WM: checkpoint folder
DINOWM_URL = "https://huggingface.co/fafraob/point-cloud-tworoom/resolve/main/dino-wm"
DINOWM_DIR = HOME / "checkpoints" / "dinowm_tworoom"
for name in ["hydra.yaml", "swm_h5_norm_stats.json", "checkpoints/model_latest.pth"]:
    target = DINOWM_DIR / name
    if not target.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{DINOWM_URL}/{name}", target)

dinowm = load_dinowm_cost_model(DINOWM_DIR, device=DEVICE)
dinowm.requires_grad_(False)
import dinowm_swm.planning as dinowm_planning

dinowm_planning._autocast = lambda enabled: torch.autocast("cuda", dtype=torch.float16, enabled=bool(enabled))
dinowm.bf16 = DEVICE == "cuda"           # the flag switches the adapter's autocast on

MODELS = {
    "lewm": dict(model=lewm, transform=lewm_transform, tokens_per_frame=1, feat_dim=192,
                 encoder="vit-tiny (end-to-end)", n_eval=N_EVAL, precision="fp32"),
    "dinowm_noprop": dict(model=dinowm, transform=dinowm.img_transform, tokens_per_frame=196, feat_dim=384,
                          encoder="dinov2_vits14 (frozen)", n_eval=N_EVAL_DINOWM,
                          precision="fp16" if dinowm.bf16 else "fp32"),
}
for name, m in MODELS.items():
    n_params = sum(p.numel() for p in m["model"].parameters()) / 1e6
    print(f"{name:14s} {n_params:5.1f} M parameters | {m['tokens_per_frame']:3d} tokens per frame | "
          f"{m['precision']} | {m['n_eval']} episodes")

11:30:29 | INFO  | utils.py    | Downloading quentinll/lewm-tworooms from HuggingFace...
11:30:29 | INFO  | utils.py    | Fetching https://huggingface.co/quentinll/lewm-tworooms/resolve/main/config.json


config.json: 100%|██████████| 1.31k/1.31k [00:00<00:00, 7.14MB/s]

11:30:29 | INFO  | utils.py    | Fetching https://huggingface.co/quentinll/lewm-tworooms/resolve/main/weights.pt



weights.pt: 100%|██████████| 72.3M/72.3M [00:01<00:00, 63.6MB/s]

11:30:31 | INFO  | utils.py    | Loading checkpoint from folder /content/stablewm/checkpoints/models--quentinll--lewm-tworooms...


11:30:43 | INFO  | __init__.py | TensorFlow version 2.21.0 available.
11:30:43 | INFO  | __init__.py | JAX version 0.11.1 available.
11:30:45 | INFO  | atomic_chec~| [atomic_save] installed crash-safe checkpoint plugin (write to sibling .tmp + fsync + atomic rename)
11:30:45 | INFO  | utils.py    | Created ViT-tiny from scratch with config: {'hidden_size': 192, 'num_hidden_layers': 12, 'num_attention_heads': 3, 'intermediate_size': 768, 'image_size': 224, 'patch_size': 14}
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip
11:30:52 | INFO  | __init__.py | using MLP layer as FFN
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 239MB/s]


num_action_repeat: 1
num_proprio_repeat: 1
proprio encoder: ProprioceptiveEmbedding(
  (patch_embed): Conv1d(1, 10, kernel_size=(1,), stride=(1,))
)
action encoder: ProprioceptiveEmbedding(
  (patch_embed): Conv1d(10, 10, kernel_size=(1,), stride=(1,))
)
proprio_dim: 10, after repeat: 10
action_dim: 10, after repeat: 10
emb_dim: 404
Model emb_dim:  404
[dinowm] loaded /content/stablewm/checkpoints/dinowm_tworoom/checkpoints/model_latest.pth (epoch None), encoder dinov2_vits14, num_hist 3, frameskip 5, mixed_precision bf16
lewm            18.0 M parameters |   1 tokens per frame | fp32 | 50 episodes
dinowm_noprop   42.2 M parameters | 196 tokens per frame | fp16 | 50 episodes


## 6. Evaluation function

One function for every model. It runs the pinned queries through `World.evaluate` and times every call of
the CEM solver. A solver call plans for all environments one after another, so
`time per replan = solver time / number of environments planned`.

The first replan of a run includes one-off GPU warm-up, so the median over replan rounds is reported next to
the mean.

In [7]:
from sklearn import preprocessing

process = {}
for col in ["action", "proprio"]:
    data = dataset.get_col_data(col)
    process[col] = preprocessing.StandardScaler().fit(data[~np.isnan(data).any(axis=1)])
    if col != "action":
        process[f"goal_{col}"] = process[col]

CALLABLES = [
    {"method": "_set_state", "args": {"state": {"value": "proprio"}}},
    {"method": "_set_goal_state", "args": {"goal_state": {"value": "goal_proprio"}}},
]


def sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()


def run_eval(name, n_eval, video=True):
    """Evaluate MODELS[name] on the first `n_eval` pinned queries. Returns a result dict."""
    spec = MODELS[name]
    solver = swm.solver.CEMSolver(model=spec["model"], device=DEVICE, seed=SEED, **CEM)

    solve_log = []                                   # (environments planned, seconds) per solver call
    plain_solve = solver.solve

    def timed_solve(info_dict, init_action=None):
        n_envs = len(next(iter(info_dict.values())))
        sync(); t0 = time.perf_counter()
        out = plain_solve(info_dict, init_action=init_action)
        sync(); solve_log.append((n_envs, time.perf_counter() - t0))
        return out

    solver.solve = timed_solve

    world = swm.World(ENV_NAME, num_envs=n_eval, image_shape=(224, 224), max_episode_steps=2 * EVAL_BUDGET)
    world.set_policy(swm.policy.WorldModelPolicy(
        solver=solver, config=swm.PlanConfig(**PLAN), process=process,
        transform={"pixels": spec["transform"], "goal": spec["transform"]},
    ))
    t0 = time.perf_counter()
    metrics = world.evaluate(
        dataset=dataset,
        episodes_idx=QUERIES["episodes"][:n_eval],
        start_steps=QUERIES["start_steps"][:n_eval],
        goal_offset=GOAL_OFFSET,
        eval_budget=EVAL_BUDGET,
        callables=CALLABLES,
        video=(HOME / "videos" / name) if video else None,
    )
    wall = time.perf_counter() - t0
    world.close()

    per_replan = [seconds / n for n, seconds in solve_log]
    return {
        "method": name,
        "n_episodes": n_eval,
        "success_rate": metrics["success_rate"],
        "episode_successes": np.asarray(metrics["episode_successes"]).astype(int).tolist(),
        "replans": int(sum(n for n, _ in solve_log)),
        "time_per_replan_mean_s": float(sum(s for _, s in solve_log) / sum(n for n, _ in solve_log)),
        "time_per_replan_median_s": float(np.median(per_replan)),
        "wall_time_s": wall,
    }

## 7. Short trial run and time estimate

Two episodes per model first. This checks the code path and projects how long the full run will take, so a
multi-hour DINO-WM evaluation does not come as a surprise. Change `N_EVAL_DINOWM` in section 3 if the
projection is too long for the session.

In [8]:
TRIAL = {}
for name in MODELS:
    TRIAL[name] = run_eval(name, n_eval=2, video=False)
    r = TRIAL[name]
    replans_per_episode = r["replans"] / r["n_episodes"]
    n_eval = MODELS[name]["n_eval"]
    projected = r["time_per_replan_mean_s"] * replans_per_episode * n_eval
    print(f"\n{name}: {r['time_per_replan_mean_s']:.1f} s per replan on {GPU} "
          f"-> about {projected / 60:.0f} min of planning for {n_eval} episodes "
          f"(up to twice that if episodes need a second replan)\n")

CEM solve time: 3.7161 seconds
CEM solve time: 0.7191 seconds

lewm: 1.5 s per replan on NVIDIA A100-SXM4-40GB -> about 2 min of planning for 50 episodes (up to twice that if episodes need a second replan)

CEM solve time: 90.6820 seconds

dinowm_noprop: 45.3 s per replan on NVIDIA A100-SXM4-40GB -> about 38 min of planning for 50 episodes (up to twice that if episodes need a second replan)



## 8. Full evaluation

LeWM first because it is fast. Each result is appended to `results/runs.csv` as soon as it is finished.

In [9]:
FIELDS = ["run_id", "date", "git_commit", "swm_version", "gpu", "env", "method", "encoder", "layer", "pooling",
          "tokens_per_frame", "feat_dim", "train_episodes", "cem_samples", "cem_iters", "horizon",
          "precision", "compiled", "seed", "n_episodes", "success_rate", "time_per_replan_median_s",
          "time_encode_s", "time_predictor_s", "time_cem_other_s", "notes"]
RUNS_CSV = RESULTS_DIR / ("runs_smoke.csv" if SMOKE else "runs.csv")


def log_run(result):
    spec = MODELS[result["method"]]
    row = dict.fromkeys(FIELDS, "")
    row.update(
        run_id=f"{result['method']}_tworoom_seed{SEED}_{datetime.datetime.now():%Y%m%d-%H%M%S}",
        date=f"{datetime.date.today()}", swm_version=version("stable-worldmodel"), gpu=GPU, env="tworoom",
        method=result["method"], encoder=spec["encoder"], layer="last", pooling="none",
        tokens_per_frame=spec["tokens_per_frame"], feat_dim=spec["feat_dim"],
        cem_samples=CEM["num_samples"], cem_iters=CEM["n_steps"], horizon=PLAN["horizon"],
        precision=spec["precision"], compiled=False, seed=SEED, n_episodes=result["n_episodes"],
        success_rate=result["success_rate"],
        time_per_replan_median_s=round(result["time_per_replan_median_s"], 3),
        notes=f"dataset={DATASET}; mean_replan_s={result['time_per_replan_mean_s']:.3f}; "
              f"wall_s={result['wall_time_s']:.0f}; replans={result['replans']}",
    )
    new_file = not RUNS_CSV.exists()
    with RUNS_CSV.open("a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDS)
        if new_file:
            writer.writeheader()
        writer.writerow(row)


RESULTS = {}
for name in MODELS:
    RESULTS[name] = run_eval(name, n_eval=MODELS[name]["n_eval"])
    log_run(RESULTS[name])
    r = RESULTS[name]
    print(f"\n{name}: success {r['success_rate']:.0f}% | {r['time_per_replan_median_s']:.2f} s per replan "
          f"(median) | wall {r['wall_time_s'] / 60:.1f} min\n")

CEM solve time: 35.5582 seconds
CEM solve time: 5.6931 seconds

lewm: success 86% | 0.71 s per replan (median) | wall 1.0 min

CEM solve time: 2248.7763 seconds
CEM solve time: 89.9720 seconds

dinowm_noprop: success 100% | 44.98 s per replan (median) | wall 39.3 min



## 9. Summary

In [10]:
import pandas as pd

table = pd.DataFrame([
    {"method": name, "tokens per frame": MODELS[name]["tokens_per_frame"], "episodes": r["n_episodes"],
     "success %": r["success_rate"], "s per replan (median)": round(r["time_per_replan_median_s"], 2),
     "s per replan (mean)": round(r["time_per_replan_mean_s"], 2), "wall min": round(r["wall_time_s"] / 60, 1)}
    for name, r in RESULTS.items()
]).set_index("method")
ratio = RESULTS["dinowm_noprop"]["time_per_replan_median_s"] / RESULTS["lewm"]["time_per_replan_median_s"]
pairs = list(zip(RESULTS["lewm"]["episode_successes"], RESULTS["dinowm_noprop"]["episode_successes"]))
print(f"GPU: {GPU} | dataset: {DATASET} | seed {SEED} | CEM {CEM}")
print(f"DINO-WM needs {ratio:.0f}x the planning time of LeWM per replan")
print(f"on the {len(pairs)} episodes both models ran: LeWM solves {sum(a for a, _ in pairs)}, "
      f"DINO-WM solves {sum(b for _, b in pairs)}")
print("rows appended to", RUNS_CSV)
table

GPU: NVIDIA A100-SXM4-40GB | dataset: tworoom | seed 42 | CEM {'num_samples': 300, 'n_steps': 30, 'topk': 30}
DINO-WM needs 63x the planning time of LeWM per replan
on the 50 episodes both models ran: LeWM solves 43, DINO-WM solves 50
rows appended to /content/results/runs.csv


,tokens per frame,episodes,success %,s per replan (median),s per replan (mean),wall min
method,,,,,,
lewm,1,50,86.0,0.71,0.71,1.0
dinowm_noprop,196,50,100.0,44.98,44.98,39.3


## Notes for reading the numbers

- One seed, and DINO-WM runs on fewer episodes than LeWM (the first `N_EVAL_DINOWM` of the same queries), so
  its success rate is coarse. The time per replan is the reliable part.
- The DINO-WM checkpoint is a third-party retrain; its authors report 100% on TwoRoom at seed 42.
- Precision differs between the two models (see section 5) and is recorded per row in `runs.csv`.
- Videos of every episode are in `$STABLEWM_HOME/videos/<method>/`.
- On Colab, `results/` lives on the session disk. Download `runs.csv` and the query file before the session ends.